In [ ]:
!pip install pandas pyarrow sentence-transformers scipy numpy

In [ ]:
import re, random, itertools
from collections import Counter
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sentence_transformers import SentenceTransformer

BASE = "https://huggingface.co/datasets/allenai/scico/resolve/refs%2Fconvert%2Fparquet/default"
splits = {s: pd.read_parquet(f"{BASE}/{s}/0000.parquet") for s in ["train", "validation", "test"]}
for s, d in splits.items():
    print(s, d.shape)
print(splits["train"].columns.tolist())

train (221, 13)
validation (100, 13)
test (200, 13)
['flatten_tokens', 'flatten_mentions', 'tokens', 'doc_ids', 'metadata', 'sentences', 'mentions', 'relations', 'id', 'source', 'hard_10', 'hard_20', 'curated']


In [ ]:
END_INCLUSIVE = True

def extract_mentions(df):
    rows = []
    for topic_id, r in df.reset_index(drop=True).iterrows():
        toks = list(r["flatten_tokens"])
        for start, end, cid in r["flatten_mentions"]:
            end_excl = end + 1 if END_INCLUSIVE else end
            rows.append((topic_id, int(start), int(end), int(cid), " ".join(toks[start:end_excl])))
    return pd.DataFrame(rows, columns=["topic", "start", "end", "gold", "text"])

def normalize(t):
    t = t.lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

M = {}
for s, d in splits.items():
    M[s] = extract_mentions(d)
    M[s]["norm"] = M[s]["text"].map(normalize)
    print(s, "topics:", M[s].topic.nunique(), "mentions:", len(M[s]),
          "gold clusters:", M[s].groupby("topic").gold.nunique().sum())
M["train"].sample(8, random_state=0)

train topics: 221 mentions: 10922 gold clusters: 4079
validation topics: 100 mentions: 4873 gold clusters: 1747
test topics: 200 mentions: 10423 gold clusters: 3711


,topic,start,end,gold,text,norm
1534,29,2782,2782,12,ST-DQN,st dqn
5233,106,1134,1136,11,handwritten Chinese recognition,handwritten chinese recognition
3655,73,1658,1660,6,multi-scale training strategy,multi scale training strategy
6275,127,254,256,20,3D full-shape reconstruction,3d full shape reconstruction
5407,109,1024,1027,4,maxout and fuse operations,maxout and fuse operations
2164,43,442,444,4,depth estimation step,depth estimation step
7122,143,3464,3466,7,online crowdsourcing tools,online crowdsourcing tools
5429,109,2346,2349,9,Sign and PReLU functions,sign and prelu functions


In [ ]:
def n_pairs(n): return n * (n - 1) // 2

def pair_counts(pred, gold):
    """TP, #predicted pairs, #gold pairs for one topic."""
    pred, gold = np.asarray(pred), np.asarray(gold)
    tp = sum(n_pairs(c) for c in Counter(zip(pred, gold)).values())
    pp = sum(n_pairs(c) for c in Counter(pred).values())
    gp = sum(n_pairs(c) for c in Counter(gold).values())
    return tp, pp, gp

def prf(counts):
    tp = sum(c[0] for c in counts); pp = sum(c[1] for c in counts); gp = sum(c[2] for c in counts)
    p = tp / pp if pp else 0.0
    r = tp / gp if gp else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

In [ ]:
def exact_labels(df):
    return {t: g["norm"].factorize()[0] for t, g in df.groupby("topic")}

def eval_labels(df, labels):
    return prf([pair_counts(labels[t], g["gold"].values) for t, g in df.groupby("topic")])

exact = {}
for s in ["validation", "test"]:
    exact[s] = eval_labels(M[s], exact_labels(M[s]))
    print(f"Exact  {s:10s} P={exact[s][0]:.4f} R={exact[s][1]:.4f} F1={exact[s][2]:.4f}")

Exact  validation P=0.9147 R=0.0691 F1=0.1285
Exact  test       P=0.9144 R=0.0623 F1=0.1166


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
all_strings = sorted(set().union(*[set(M[s]["norm"]) for s in ["validation", "test"]]))
E = model.encode(all_strings, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
idx = {s: i for i, s in enumerate(all_strings)}

def topic_sims(df):
    out = {}
    for t, g in df.groupby("topic"):
        e = E[[idx[x] for x in g["norm"]]]
        out[t] = e @ e.T
    return out

SIM = {s: topic_sims(M[s]) for s in ["validation", "test"]}

def cluster_by_threshold(S, thr):
    return connected_components(csr_matrix(S >= thr), directed=False)[1]

def eval_threshold(split, thr):
    df = M[split]
    return prf([pair_counts(cluster_by_threshold(SIM[split][t], thr), g["gold"].values)
                for t, g in df.groupby("topic")])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

In [ ]:
thresholds = np.round(np.arange(0.50, 1.0001, 0.01), 2)
val_curve = pd.DataFrame([(t, *eval_threshold("validation", t)) for t in thresholds],
                         columns=["threshold", "precision", "recall", "f1"])
best_thr = float(val_curve.loc[val_curve.f1.idxmax(), "threshold"])
print(val_curve.iloc[::3].to_string(index=False))
print("\nBest validation threshold:", best_thr, "| val F1 =", round(val_curve.f1.max(), 4))

 threshold  precision   recall       f1
      0.50   0.178931 0.988983 0.303036
      0.53   0.189754 0.985272 0.318221
      0.56   0.202926 0.973855 0.335866
      0.59   0.225722 0.969517 0.366188
      0.62   0.241467 0.961126 0.385966
      0.65   0.265728 0.957472 0.416002
      0.68   0.286101 0.947882 0.439536
      0.71   0.317951 0.924649 0.473190
      0.74   0.356062 0.886802 0.508111
      0.77   0.411102 0.847642 0.553675
      0.80   0.473800 0.773205 0.587559
      0.83   0.560552 0.686722 0.617255
      0.86   0.648410 0.581859 0.613334
      0.89   0.713968 0.439434 0.544028
      0.92   0.820878 0.279655 0.417185
      0.95   0.888588 0.131122 0.228523
      0.98   0.918125 0.074894 0.138492

Best validation threshold: 0.85 | val F1 = 0.6203


In [ ]:
emb_test = eval_threshold("test", best_thr)
results = pd.DataFrame({
    "Exact normalized string": exact["test"],
    f"Embedding (thr={best_thr})": emb_test,
}, index=["precision", "recall", "f1"]).T.round(4)
results

,precision,recall,f1
Exact normalized string,0.9144,0.0623,0.1166
Embedding (thr=0.85),0.5858,0.5364,0.5600


In [ ]:
test_pred = {t: cluster_by_threshold(SIM["test"][t], best_thr) for t in SIM["test"]}
test_exact = exact_labels(M["test"])

def show_clusters(topic, labels, max_clusters=8, max_items=6):
    g = M["test"][M["test"].topic == topic].copy()
    g["pred"] = labels[topic]
    print(f"Topic {topic}: {len(g)} mentions -> {g.pred.nunique()} predicted clusters (gold: {g.gold.nunique()})")
    for c, grp in list(g.groupby("pred"))[:max_clusters]:
        items = list(dict.fromkeys(grp["text"]))[:max_items]
        print(f"  cluster {c}: {items}")

for t in sorted(test_pred)[:2]:
    print("--- embedding ---"); show_clusters(t, test_pred)
    print("--- exact ---");     show_clusters(t, test_exact)

--- embedding ---
Topic 0: 42 mentions -> 24 predicted clusters (gold: 26)
  cluster 0: ['batch normalization', 'Batch Normalization', 'Batch Normalization configuration', 'batch normalization process', 'batch - normalization', 'Batch Normalization Layers']
  cluster 1: ['channel normalization', 'channel based normalization', 'channel normalization filter', 'channel normalization layer', 'channel normalization steps']
  cluster 2: ['BatchChannel Normalization ( BCN )']
  cluster 3: ['switchable normalization', 'Switchable Normalization']
  cluster 4: ['Switchable Normalization ( SN )']
  cluster 5: ['switch normalization layer']
  cluster 6: ['moment-based normalization algorithm', 'moment-based normalization']
  cluster 7: ['Weighted Data Normalization method ( WDN )']
--- exact ---
Topic 0: 42 mentions -> 33 predicted clusters (gold: 26)
  cluster 0: ['batch normalization', 'Batch Normalization', 'batch - normalization']
  cluster 1: ['channel normalization']
  cluster 2: ['channel b

In [ ]:
def collect_errors(df, labels, sims=None):
    missed, merged = [], []
    for t, g in df.groupby("topic"):
        txt, gold, pred = g["text"].values, g["gold"].values, labels[t]
        for i, j in itertools.combinations(range(len(g)), 2):
            if txt[i].lower() == txt[j].lower():
                same_text = True
            else:
                same_text = False
            s = float(sims[t][i, j]) if sims is not None else None
            if gold[i] == gold[j] and pred[i] != pred[j]:
                missed.append((t, txt[i], txt[j], s))
            elif gold[i] != gold[j] and pred[i] == pred[j]:
                merged.append((t, txt[i], txt[j], s))
    cols = ["topic", "mention_a", "mention_b", "cosine"]
    return pd.DataFrame(missed, columns=cols), pd.DataFrame(merged, columns=cols)

def report(name, labels, sims=None, k=12):
    missed, merged = collect_errors(M["test"], labels, sims)
    print(f"===== {name}: {len(missed)} missed matches, {len(merged)} false merges =====")
    for label, d in [("MISSED MATCHES (gold-coreferent, not linked)", missed),
                     ("FALSE MERGES (different gold clusters, linked)", merged)]:
        print("\n" + label)
        if len(d):
            pd.set_option("display.max_colwidth", 60)
            print(d.sample(min(k, len(d)), random_state=42).to_string(index=False))
    return missed, merged

emb_missed, emb_merged = report(f"Embedding (thr={best_thr})", test_pred, SIM["test"])

===== Embedding (thr=0.85): 18281 missed matches, 14958 false merges =====

MISSED MATCHES (gold-coreferent, not linked)
 topic                                mention_a                                       mention_b   cosine
    32      hyperparameter optimization process                          Parameter optimization 0.715189
   156                gradient descent strategy                             gradient update law 0.525662
    86                                    GloVe                                GloVe embeddings 0.680641
   117              heterogeneous compute nodes              heterogeneous compute architecture 0.795285
   102                     Leaky ReLU ( LReLU )                        leaky ReLU parameterized 0.688778
    59                         GPS enhancements                                      GPS metric 0.671613
   153                                     ReLU                  Rectified Linear Unit networks 0.266013
   133                     video 's des

In [ ]:
exact_missed, exact_merged = report("Exact normalized string", test_exact)

===== Exact normalized string: 36976 missed matches, 230 false merges =====

MISSED MATCHES (gold-coreferent, not linked)
 topic                                 mention_a                              mention_b cosine
    19                   Linear Layer Operations                   linear layered model   None
     3                 BERT pre-trained baseline                             BERT model   None
   189                     outper-forming PSPNet            PSPNet and DeepLabV3 models   None
    97 supervised per-pixel image classification     per - pixel classification problem   None
    52           outdoor illumination estimation                  outdoor light capture   None
    90          Distributed word representations distributed ( output ) word embeddings   None
   118                      Deep learning models               deep learning algorithms   None
    68               multi-task learning problem            multi-task learning setting   None
   108                 

In [ ]:
for name, d in [("missed", emb_missed), ("false merges", emb_merged)]:
    c = Counter(zip(d.mention_a.str.lower(), d.mention_b.str.lower())).most_common(10)
    print(f"Most frequent {name}:")
    for (a, b), n in c: print(f"  {n:4d}  {a!r} <-> {b!r}")

Most frequent missed:
    14  'convolutional neural network architectures' <-> 'convolutional neural networks'
    14  'googlenet' <-> 'googlenet model'
    13  'tanh' <-> 'tanh activation'
    12  'glove' <-> 'glove embeddings'
    12  'vae' <-> 'variational autoencoder architecture'
    12  'esim' <-> 'enhanced sequential inference model'
    12  'sentence embedding' <-> 'sentence encoding'
    12  'object recognition' <-> 'recognizing objects'
    10  'convolutional neural network architectures' <-> 'convolutional networks'
    10  'sentiment analysis' <-> 'sentiment classification'
Most frequent false merges:
    20  'representation learning' <-> 'unsupervised representation learning'
    15  'domain adaptation setting' <-> 'unsupervised domain adaptation'
    15  'face recognition' <-> 'face verification'
    13  'information extraction' <-> 'extracting information'
    12  'semantic parsing' <-> 'dependency parsing'
    12  'domain adaptation problem' <-> 'unsupervised domain ada